# 검색 실험실 (Step 3) — 구성을 바꿔 가며 같은 질의를 나란히 비교

구성 = `configs/retrieval/<이름>.yaml` (코퍼스 · 토크나이저 · 임베더 · 검색기 · 합치기 · 재정렬 체인).
API(`/api/v1/retrieval-lab/*`)와 CLI(`eval/retrieval/compare.py`)도 같은 함수를 쓴다.

| 하는 일 | 함수 |
|---|---|
| 구성 목록·펼친 모양 | `list_configs()`, `load_config(name, overrides)` |
| 질의 하나 × 구성 하나 (단계별 순위) | `get_pipeline(cfg).search(query, filters, top_k, tracer)` |
| 질의 하나 × 구성 여러 개 (나란히) | `side_by_side(...)` (이 노트북) |
| 질의 묶음 × 구성들 → 보고서·판정표 | `lab.run_compare(lab.load_queries(...), [...])` |
| 사람 판정 → 점수 | `lab.score_review(run_id)` |

> 저장소 루트에서 실행. 데이터·HCX 없이 구조만 보려면 `V4_NB_FAKE=1` (가짜 데이터 + HCX 대역 + 가짜 표 15개).
> 로컬 모델 구성(`st_*`, `rerank_*`)은 `pip install sentence-transformers` 후 첫 실행 때 HuggingFace 에서 모델을 받는다 (GPU 있으면 GPU).

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parents[1]   # notebooks/v4 에서 열어도
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
USE_FAKE = os.getenv("V4_NB_FAKE", "0") == "1"
if USE_FAKE:
    import tempfile
    from tests import fake_kosis
    from tests.v4.test_retrieval_lab import _extra_tables
    tmp = Path(tempfile.mkdtemp()) / "storage"
    fake_kosis.TABLES[:] = fake_kosis.TABLES + _extra_tables()
    fake_kosis.build(tmp / "kosis_data")
    for k in ("KOSIS_DATA_DIR", "KOSIS_DATA", "VIEW_OUT_DIR", "TABLE_INDEX_DIR", "VIEW_EMBED_CACHE", "CSV_INDEX_DIR", "RUNS_DIR"):
        os.environ[k] = ""
    os.environ["STORAGE_DIR"], os.environ["HCX_API_KEY"] = str(tmp), "test"
    import backend.core.llm as llm, backend.rag.retrieval.dense as dense
    llm._post, dense._post = fake_kosis.fake_hcx_post, fake_kosis.fake_dense_post
    from backend.rag.ingestion import meta_profile, table_text
    meta_profile.build_all(); table_text.build_index(progress=lambda m: None)
    from backend.rag.csv_index import build_index
    build_index(scope="all", log=lambda m: None)
import pandas as pd
pd.set_option("display.max_colwidth", 60)
from backend.core.config import TABLE_INDEX_DIR, RUNS_DIR
from backend.rag.retrieval import factory, lab
from backend.rag.retrieval.factory import get_pipeline, load_config, list_configs
print("표 색인:", TABLE_INDEX_DIR, "| 실행 기록:", RUNS_DIR)

## 1. 구성 목록 — `missing` 이 있으면 그 패키지 설치 필요

In [ ]:
pd.DataFrame(list_configs())[['name', 'description', 'retrievers', 'rerank', 'missing']]

## 2. 질의 하나 × 구성 하나 — 단계별 순위

`hits[].stages` 에 단계(검색기·합치기·재정렬)마다 몇 위였는지가 남는다. `trace` 는 단계별 파라미터·출력·초.
검색 문장은 `views.run_query` 처럼 `resolve.search_query` 로 다듬는다 (지역·시점 단어 빼고 지표·대상 더함).

In [ ]:
from backend.core.tracking import Tracer
from backend.rag.ingestion.resolve import search_query
F = {"시점": [2025], "지역": ["전국"], "대상": ["1인가구"], "지표": ["가구수"], "의도": "값 조회"}
Q = search_query("2025년 전국 1인가구 가구수", F)
tr = Tracer("nb")
r = get_pipeline("baseline").search(Q, F, top_k=5, tracer=tr)
print("검색 문장:", r["search_query"], "|", r["seconds"], "초")
pd.DataFrame([{**{k: h[k] for k in ("rank", "tbl_id", "tbl_nm", "score", "cost")},
               **{f"{s}#": v["rank"] for s, v in h["stages"].items()}} for h in r["hits"]])

In [ ]:
pd.DataFrame(r['trace'])[['step', 'params', 'output', 'seconds']]

## 3. 질의 하나 × 구성 여러 개 — 나란히

In [ ]:
def side_by_side(query, filters=None, configs=("baseline", "bm25_only", "hybrid_rrf_hcx"), top_k=5, overrides=None):
    cols = {}
    for c in configs:                       # 이름 또는 dict 구성
        name = c if isinstance(c, str) else c.get("name", "inline")
        try:
            hs = get_pipeline(c, (overrides or {}).get(name)).search(query, filters, top_k)["hits"]
            cols[name] = [f"{h['tbl_nm'][:22]} ({h['tbl_id']})" for h in hs] + [""] * (top_k - len(hs))
        except Exception as e:
            cols[name] = [f"오류: {e}"] + [""] * (top_k - 1)
    return pd.DataFrame(cols, index=range(1, top_k + 1))

side_by_side(Q, F, ["baseline", "bm25_only", "kiwi_lexicon", "char_ngram", "hybrid_rrf_hcx", "ensemble_rrf"])

## 4. 파라미터 바꿔 보기 — 점 경로 overrides / dict 구성

- `retrievers.0.k`: BM25 1단계 후보 수 · `rerank.0.top_n`: 조건 확인 뒤 남길 수 · `fusion.weights` · `top_k`
- 새 구성은 dict 로 바로: `extends` 로 기존 구성을 물려받고 바꿀 칸만

In [ ]:
cfg = load_config("hybrid_rrf_hcx", {"fusion.weights": {"bm25": 1.0, "dense": 2.0}, "retrievers.1.k": 30})
mine = {"name": "bm25plus_char", "extends": "baseline",
        "retrievers": [{"name": "bm25", "type": "bm25", "variant": "plus", "k": 50, "tokenizer": {"type": "char_ngram", "n": [2, 3]}}]}
pd.concat({"dense×2": side_by_side(Q, F, [cfg], 5), "BM25+ 글자": side_by_side(Q, F, [mine], 5)}, axis=1)

## 5. 질의 묶음 × 구성들 → 보고서·판정표

질의 출처: `bundles`(재작성 80문항 결과, HCX 없음) · `golden`(질문 50개) · `examples` · yaml 경로 (`eval/retrieval/queries_sample.yaml`).
`report.html` 을 브라우저로 열면 구성별 상위 k 가 나란히 (같은 표 = 같은 색, 칸에 마우스 → 단계별 순위).

In [ ]:
qs = lab.load_queries("eval/retrieval/queries_sample.yaml")
res = lab.run_compare(qs, ["baseline", "bm25_only", "kiwi_lexicon", "hybrid_rrf_hcx"], top_k=5)
print("보고서:", res["files"]["html"]); print("판정표:", res["files"]["review"])
pd.DataFrame(res["summary"]).T

In [ ]:
from IPython.display import IFrame
IFrame(res["files"]["html"], width="100%", height=500)   # Jupyter 가 파일 경로를 못 열면 위 경로를 브라우저로

## 6. 사람 판정 → 점수

`review.csv` 는 (질의, 표) 한 줄 — 어느 구성이 몇 위로 냈는지(`found_by`)가 함께 있다. `label` 칸에
**2** = 이 표로 바로 답할 수 있음 · **1** = 관련 있음·조건 일부 · **0** = 무관 (O/△/X 도 됨). 같은 표는 한 번만 판정하면 모든 구성에 적용.
아래는 예시로 기준 구성 1위만 2 를 준 것. 실제로는 엑셀에서 채운 뒤 `lab.score_review(run_id)`.

In [ ]:
rv = pd.read_csv(res["files"]["review"], encoding="utf-8-sig", dtype=str).fillna("")
top1 = {q["id"]: q["results"]["baseline"]["hits"][0]["tbl_id"] for q in res["queries"]}
judg = [{"query_id": r.query_id, "tbl_id": r.tbl_id, "label": 2 if top1[r.query_id] == r.tbl_id else 0} for r in rv.itertuples()]
pd.DataFrame(lab.score_review(res["run_id"], judg)["by_config"]).T

## 7. 구성으로 View 까지

`POST /api/v1/rag/views?retrieval_config=<이름>` 과 같다 (지정하지 않으면 기존 검색).

In [ ]:
from backend.rag.ingestion.views import run_query
vb = {"query_id": "nb", "original": "2025년 전국 1인 가구 수", "search_text": "2025년 전국 1인가구 가구수", "filters": F}
out = run_query(vb, top_k=5, n_views=1, search_fn=get_pipeline("hybrid_rrf_hcx").search_compat)
[(v["tbl_id"], len(v["record"]["table"]["rows"]), v["notes"][:2]) for u in out["units"] for v in u["views"]]